# SupportLens walkthrough
Unstructured text → linear algebra → probability → statistics → PyTorch → agent tools.
Run top to bottom (≈1 minute on CPU). Every section ends with a **Try it** exercise.

In [ ]:
import sys

sys.path.insert(0, "..")
import numpy as np

from supportlens.data import CATEGORIES, generate_tickets
from supportlens.linalg_stats import *
from supportlens.model import TransformerTextClassifier
from supportlens.preprocess import Vocab, clean_text, tokenize
from supportlens.train import fit, predict, set_seed, split

set_seed(0)

## 1. Unstructured data
Raw tickets contain HTML, emails, URLs, order ids, typos and shouting.

In [ ]:
data = generate_tickets(3000, seed=0)
texts = [d["text"] for d in data]
y = np.array([CATEGORIES.index(d["label"]) for d in data])
for t in texts[:3]:
    print("RAW  :", t)
    print("CLEAN:", clean_text(t), "\n")

## 2. Linear algebra: TF-IDF, cosine similarity, SVD
Documents become vectors; similarity is the angle between them: $\cos\theta = \frac{a\cdot b}{\|a\|\|b\|}$.
PCA is the SVD of the centred matrix, $X_c = U\Sigma V^\top$; component $i$ explains $\sigma_i^2/\sum_j\sigma_j^2$ of the variance.

In [ ]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(texts)
print("matrix shape (docs x terms):", X.shape)
sims = cosine_similarity(X[0], X)
print("query:", texts[0][:90])
for i in np.argsort(-sims)[1:4]:
    print(f"  {sims[i]:.3f} [{data[i]['label']}] {texts[i][:80]}")

proj, comps, evr = pca_svd(X, 2)
print(
    "cumulative variance, top 1/5/10/20 components:",
    np.round(np.cumsum(evr)[[0, 4, 9, 19]], 3),
)

**Try it:** project to 2-D (`proj`) and colour by label with matplotlib. Which categories overlap?

## 3. Probability: Naive Bayes from scratch
$\hat c = \arg\max_c\ \log P(c) + \sum_w n_w \log P(w\mid c)$ with Laplace smoothing. In matrix form that is one `counts @ log_lik.T`.

In [ ]:
tr, va, te = split(len(texts), 0)
vocab = Vocab([tokenize(texts[i]) for i in tr], min_freq=2)
ids = vocab.encode_batch(texts, 40)
counts = bag_of_words(ids, len(vocab))
nb = MultinomialNaiveBayes().fit(counts[tr], y[tr], len(CATEGORIES))
nb_pred = nb.predict(counts[te])
print(
    "NB accuracy:",
    round(accuracy(y[te], nb_pred), 4),
    "| macro-F1:",
    round(macro_f1(y[te], nb_pred, 5), 4),
)

**Try it:** change `alpha` (smoothing) to 0.01 and 10. What happens to accuracy, and why?

## 4. Deep learning: Transformer encoder in PyTorch

In [ ]:
model = TransformerTextClassifier(len(vocab), len(CATEGORIES), max_len=40)
fit(
    model,
    ids[tr],
    y[tr],
    ids[va],
    y[va],
    epochs=6,
    lr=3e-4,
    batch_size=64,
    device="cpu",
)
tf_pred = predict(model, ids[te], "cpu")
print("Transformer accuracy:", round(accuracy(y[te], tf_pred), 4))

**Try it:** drop the encoder and mean-pool the embeddings only, then compare. Does attention (or position) help here?

## 5. Statistics: is the difference real?
Bootstrap gives a confidence interval; a paired permutation test asks whether the two models differ more than chance.

In [ ]:
lo, hi = bootstrap_ci(y[te], tf_pred)
print(f"Transformer accuracy 95% CI: [{lo:.3f}, {hi:.3f}]")
print(
    "p-value (Transformer vs NB):",
    paired_permutation_test(tf_pred == y[te], nb_pred == y[te]),
)

## 6. Real data (Bitext customer-support set)
Downloads ~19 MB from GitHub the first time. Duplicates are dropped before splitting so the same sentence cannot sit in both train and test.

In [ ]:
from supportlens.dataset_io import download_bitext, load_csv

try:
    real = load_csv(
        str(download_bitext("../data/bitext.csv")), "instruction", "category"
    )
    print(real[0])
except Exception as e:
    print("download failed (offline?):", e)

**Try it:** rerun sections 3 to 5 on `real` instead of `data`. Does attention still beat Naive Bayes? (Spoiler from the CLI run: no. Work out why.)

> **Caveat:** the data here is templated, so scores are optimistic. Swap `generate_tickets` for a real
> dataset (e.g. a public support-ticket or news-category CSV) before quoting any number.